## this script calculates the embedding similarity of papers by differentiate them by domestic, international, international-first, international-middle, international-last

In [25]:
import pickle
# %%
import sys

import numpy as np
import pandas as pd
from scipy import sparse
from sentence_transformers import SentenceTransformer, models
from tqdm import tqdm
import os

In [33]:

paper_table_file = "../../data/dropbox/LaborDivision/Data/cleanedraw/pub_title.csv"
sbert_model_file = "/gpfs/sciencegenome/NSF-TIP/derived/models/model~specter_dim~64.sav"
device = "cuda:0"
output_file = "../../data/dropbox/LaborDivision/Data/deriveddata/embedding/wos_specter_dim64"


In [6]:
#
# Load
#
model = SentenceTransformer(sbert_model_file)
paper_table = pd.read_csv(paper_table_file)


In [ ]:
# Drop papers without titles
#
df = paper_table[["paper_id", "title"]].dropna()
paper_ids = df["paper_id"].astype(int).values
titles = df["title"].values

In [11]:
# %%
# Embedding
#
emb = model.encode(
    titles,
    batch_size=1024,
    show_progress_bar=True,
)

Batches: 100%|██████████████████████████| 16481/16481 [2:08:55<00:00,  2.13it/s]


In [ ]:
# Save
#
U = sparse.csr_matrix(
    (np.ones(len(paper_ids)), (paper_ids, np.arange(len(paper_ids)))),
    shape=(paper_table.shape[0], len(paper_ids)),
)
emb = U @ emb




In [41]:
np.save(
    output_file,
    emb,
)